# Importing the Packages

In [2]:
!pip3 install groq chromadb python-dotenv

# Load the Packages

In [3]:
import os
import json
from groq import Groq
from dotenv import load_dotenv
import chromadb

In [4]:
from google.colab import userdata
groqapikey = userdata.get('GROQ_API_KEY')

In [5]:
client = Groq(api_key=groqapikey)
print("OpenAI client created successfully!!")

OpenAI client created successfully!!


# **Communicating to api**

In [6]:
for m in client.models.list().data:
    print(m.id)

meta-llama/llama-prompt-guard-2-22m
allam-2-7b
qwen/qwen3.8-27b
whisper-large-v3-turbo
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-20b
whisper-large-v3


In [7]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "what is RAG ? Explain in 3 bullet points?"},])
print(response.choices[0].message.content)

- **Hybrid architecture**: RAG fuses a retrieval system (e.g., dense or sparse search over a document corpus) with a generative language model, so the model can fetch relevant facts before forming a response.  
- **Dynamic knowledge integration**: Unlike static fine‑tuned models, RAG pulls up‑to‑date or domain‑specific documents at inference time, reducing hallucinations and keeping answers grounded in real data.  
- **End‑to‑end trainability**: The retrieval component can be jointly optimized with the generator, enabling the system to learn which passages are most useful for a given query while still producing fluent, coherent text.


#  Loading and Chunking data from the documents

In [8]:
with open("/content/drive/MyDrive/data/company_hr_policy.txt", "r") as f:
     hr_policy = f.read()
with open("/content/drive/MyDrive/data/engineering_standards.txt", "r") as f:
  engineering_document = f.read()

with open("/content/drive/MyDrive/data/onboarding_guide.txt", "r") as f:
  onboarding = f.read()

with open("/content/drive/MyDrive/data/product_knowledge_base.txt", "r") as f:
  product = f.read()

with open("/content/drive/MyDrive/data/security_policy.txt", "r") as f:
  security = f.read()
print(security)

NovaTech Solutions — Information Security Policy
Classification: Internal | Version 4.1 | Effective: January 2026

SECTION 1: PASSWORD AND AUTHENTICATION

Password Requirements:
All passwords must meet these criteria:
- Minimum 12 characters
- At least one uppercase letter
- At least one lowercase letter
- At least one number
- At least one special character (!@#$%^&*)
- Cannot reuse the last 5 passwords
- Must be changed every 90 days

Two-Factor Authentication (2FA):
2FA is mandatory for all employees on:
- Company email
- GitHub
- AWS Console
- VPN
- HR portal (for accessing payslips and personal data)
Use Google Authenticator or Authy as your 2FA app. SMS-based 2FA is not accepted due to SIM-swap vulnerabilities.

Session Management:
- Idle sessions timeout after 30 minutes for sensitive systems (AWS, HR portal)
- General sessions (Slack, Jira) timeout after 8 hours
- Always lock your laptop when leaving your desk (Cmd+L on Mac)
- Never leave your laptop unattended in public spaces

In [9]:
print(f"HR policy document : {len(hr_policy)}")
print(f"HR policy Words : {len(hr_policy.split())}")

HR policy document : 7464
HR policy Words : 1052


# Chunking Strategy

In [10]:
def chunk_document(text, source_name):
    """
    Split a document into chunks by paragraph.
    Returns a list of dicts with text and metadata.
    """
    # Split on double newlines
    paragraphs = text.strip().split("\n\n")

    chunks = []
    for para in paragraphs:
        para = para.strip()
        # Skip short lines (headers, separators)
        if len(para) < 50:
            continue
        # Skip separator lines
        if para.startswith("===="):
            continue
        chunks.append({
            "text": para,
            "source": source_name
        })

    return chunks

In [11]:
hr_chunks = chunk_document(hr_policy, "HR Policy")
engineering_chunks = chunk_document(engineering_document, "Engineering")
onboarding_chunks = chunk_document(onboarding, "Onboarding")
product_chunks = chunk_document(product, "Product")
security_chunks = chunk_document(security, "Security")

In [12]:
print(len(hr_chunks))
print(len(engineering_chunks))

25
18


In [13]:
all_chunks = hr_chunks + engineering_chunks + onboarding_chunks + product_chunks + security_chunks
print(len(all_chunks))

120


# Embedding Manually

In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = ''' Office Network:
- Connect only to the "NovaTech-Secure" WiFi network (WPA3)
- The "NovaTech-Guest" network is for visitors only — employees should not use it
- Do not connect personal IoT devices (smart speakers, etc.) to the office network '''

embeddings = model.encode(sentences)

print(embeddings.shape)
print(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(384,)
[-2.71274522e-02  3.71070579e-02  2.69963648e-02 -3.76516432e-02
 -1.24880141e-02  3.69489342e-02  2.72595212e-02 -9.89013687e-02
  1.21910451e-02 -5.96426195e-03 -2.40616184e-02  9.83860716e-02
 -1.46336211e-02 -7.14525115e-03 -9.63099953e-03 -3.73131176e-03
  4.77994904e-02 -6.60055950e-02  8.68471265e-02  5.59232198e-02
  6.26666099e-02 -2.16602795e-02  5.59136644e-03 -4.52482216e-02
  1.12407371e-01  1.82983223e-02  1.06359925e-02  8.20038021e-02
 -1.59085672e-02  1.81507748e-02 -2.22688392e-02  1.61229190e-03
  2.39930465e-04  4.42378670e-02 -4.55207527e-02 -1.53136328e-01
 -4.13969159e-03  6.60401806e-02 -2.05189046e-02 -1.09434910e-02
 -1.74337719e-02 -5.66976145e-02 -7.42021725e-02  4.75947820e-02
  9.90143139e-03 -3.26359086e-02 -6.63817152e-02 -5.27318902e-02
  1.69445630e-02  1.43104633e-02  1.61807705e-02 -7.82664269e-02
 -6.00268357e-02  1.23774938e-01  2.48117596e-02 -8.87978747e-02
  8.00592452e-03  2.06095278e-02  3.16744410e-02 -4.53798324e-02
  7.19641522e-02 -

# Cosine Similarity

In [15]:
sentence1 = ''' I love chicken Biriyani '''
sentence2 = ''' I love eating flaovured rice with chicken'''
sentence3 = ''' I am doing work from Home'''

embedding1 = model.encode(sentence1)
embedding2 = model.encode(sentence2)
embedding3 = model.encode(sentence3)

print(embedding1.shape)
print(embedding2.shape)
print(embedding3.shape)

(384,)
(384,)
(384,)


In [16]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity1 = cosine_similarity([embedding1], [embedding2])
similarity2 = cosine_similarity([embedding1], [embedding3])
similarity3 = cosine_similarity([embedding2], [embedding3])

print(similarity1)
print(similarity2)
print(similarity3)

[[0.56787026]]
[[0.01995531]]
[[0.10406436]]


# Storing Chunks with Chroma DB

In [17]:
chroma_clinet = chromadb.Client()
collection = chroma_clinet.create_collection(name="company_docs")

In [18]:
documents = []
ids = []
metadatas = []

for i, chunk in enumerate(all_chunks):
  documents.append(chunk['text'])
  ids.append(f"chunk_{i}")
  metadatas.append({"source": chunk["source"]})

print(documents[110])
print(ids[110])
print(metadatas[110])

Office Network:
- Connect only to the "NovaTech-Secure" WiFi network (WPA3)
- The "NovaTech-Guest" network is for visitors only — employees should not use it
- Do not connect personal IoT devices (smart speakers, etc.) to the office network
chunk_110
{'source': 'Security'}


In [19]:
collection.add(
    documents = documents,
    ids = ids,
    metadatas = metadatas
)

In [20]:
print(f" Stored {len(documents)} chunks in chroma db")
print(f" Sources: 5 differnt files")

 Stored 120 chunks in chroma db
 Sources: 5 differnt files


# Retrievel Pipeline

In [21]:
def retrieve(question, n_results = 3):
  results = collection.query(
      query_texts = [question],
      n_results = n_results
  )
  return results['documents'][0], results['metadatas'][0]

In [22]:
chunks, sources = retrieve("What is the work from home policy?", 3)

for i in range(len(chunks)):
  print(f"--- Chunks {i+1} ---")
  print(f"Source: {sources[i]}")
  print(f"Text: {chunks[i]} ")
  print()

--- Chunks 1 ---
Source: {'source': 'HR Policy'}
Text: Eligibility:
All employees who have completed their probation period (6 months) are eligible for Work From Home (WFH) arrangements. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval. 

--- Chunks 2 ---
Source: {'source': 'HR Policy'}
Text: NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026 

--- Chunks 3 ---
Source: {'source': 'HR Policy'}
Text: Regular WFH:
Employees may work from home up to 2 days per week. The preferred WFH days are Wednesday and Friday, though teams may adjust based on project needs. Employees must be available during core working hours (10:00 AM to 6:00 PM IST) on WFH days. 



In [23]:
def ask_rag(question, n_results = 3, verbose = True):
  chunks, sources = retrieve(question, n_results)

  if verbose:
    print(f"\n{'═' * 60}")
    print(f"❓ Question: {question}")
    print(f"{'─' * 60}")
    print(f"📄 Retrieved {len(chunks)} chunks:")
    for i, (chunk, source) in enumerate(zip(chunks, sources)):
        print(f"   [{source['source']}] {chunk[:80]}...")
    print(f"{'─' * 60}")


  context = "\n\n".join(chunks)

  messages= [
      {
          "role": "system",
          "content" : (
            "You are a helpful assistance that asnwers quesitosn based ONLY on the provided context"
            "If the context does not contain informaiton to asnwer this quesitons, "
            "say I dont have enough context or information to answer this question"
            "Don not make up information or assume anything. Strickly answer only from the providede context"
          )
      },
      {
          "role": "user",
          "content": f"Context: {context}\n\n---\n\nQuestion: {question}"
      }
  ]

  response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
    temperature = 0.2 # Low temperature for actual answers. not for creativity.
    )

  answer = response.choices[0].message.content

  if verbose:
    print(f" Answer: {answer}")
    print(f"{'═' * 60}")

  return answer

print("Rag Pipeline is built ")


Rag Pipeline is built 


# Test the RAG Pipeline

In [24]:
ask_rag("What is Home Policy?")


════════════════════════════════════════════════════════════
❓ Question: What is Home Policy?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [HR Policy] Eligibility:
All employees who have completed their probation period (6 months) ...
   [Security] NovaTech Solutions — Information Security Policy
Classification: Internal | Vers...
   [Security] Company Laptops:
- Full disk encryption must be enabled (FileVault on Mac, BitLo...
────────────────────────────────────────────────────────────
 Answer: I don’t have enough context or information to answer this question.
════════════════════════════════════════════════════════════


'I don’t have enough context or information to answer this question.'

In [25]:
ask_rag("What is the work from Home Policy?")


════════════════════════════════════════════════════════════
❓ Question: What is the work from Home Policy?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [HR Policy] Eligibility:
All employees who have completed their probation period (6 months) ...
   [HR Policy] NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: J...
   [HR Policy] Regular WFH:
Employees may work from home up to 2 days per week. The preferred W...
────────────────────────────────────────────────────────────
 Answer: **Work‑From‑Home (WFH) Policy – NovaTech Solutions (Version 3.2, Jan 2026)**  

| Item | Details |
|------|---------|
| **Eligibility** | • All employees who have completed their 6‑month probation period are eligible for WFH arrangements.  <br>• Employees still in probation may request WFH only in exceptional circumstances, and only with approval from their manager and HR. |
| **Regular WFH** | • Employees may work from home up to **2 da

'**Work‑From‑Home (WFH) Policy – NovaTech Solutions (Version\u202f3.2, Jan\u202f2026)**  \n\n| Item | Details |\n|------|---------|\n| **Eligibility** | • All employees who have completed their 6‑month probation period are eligible for WFH arrangements.  <br>• Employees still in probation may request WFH only in exceptional circumstances, and only with approval from their manager and HR. |\n| **Regular WFH** | • Employees may work from home up to **2 days per week**.  <br>• The preferred WFH days are **Wednesday and Friday**, though teams may adjust the days to meet project needs. |\n| **Availability** | • On WFH days, employees must be available during the core working hours of **10:00\u202fAM to 6:00\u202fPM IST**. |\n\nThis policy outlines who can work from home, how many days they can do so, which days are preferred, and the required availability during core hours.'

In [26]:
ask_rag("How do I cancel my subscription?")


════════════════════════════════════════════════════════════
❓ Question: How do I cancel my subscription?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [Product] "How do I cancel my subscription?":
Go to Settings → Billing → Subscription → Ca...
   [Product] Refund Policy:
Monthly subscriptions: Full refund if cancelled within 48 hours o...
   [Product] Free Trial:
All new accounts start with a 14-day free trial of the Business plan...
────────────────────────────────────────────────────────────
 Answer: To cancel your subscription, follow these steps:

1. Open the app or website and go to **Settings**.  
2. Navigate to **Billing**.  
3. Select **Subscription**.  
4. Click **Cancel**.  

You’ll be prompted to confirm the cancellation and to provide a reason. Your account will stay active until the end of the current billing period, and you can reactivate it at any time before that date.
═══════════════════════════════════════════════════════════

'To cancel your subscription, follow these steps:\n\n1. Open the app or website and go to **Settings**.  \n2. Navigate to **Billing**.  \n3. Select **Subscription**.  \n4. Click **Cancel**.  \n\nYou’ll be prompted to confirm the cancellation and to provide a reason. Your account will stay active until the end of the current billing period, and you can reactivate it at any time before that date.'

In [27]:
ask_rag("What is the capital of France?")


════════════════════════════════════════════════════════════
❓ Question: What is the capital of France?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [Product] What is CloudDesk Pro?
CloudDesk Pro is a cloud-based project management and tea...
   [Onboarding] Communication:
- Slack: Primary daily communication. Check channels: #general, #...
   [HR Policy] Communication:
Slack is the primary communication tool for daily work. Email is ...
────────────────────────────────────────────────────────────
 Answer: I don’t have enough context or information to answer this question.
════════════════════════════════════════════════════════════


'I don’t have enough context or information to answer this question.'

# Agentic RAG

In [28]:
def search_docs(query:str) ->str:
  results = collection.query(
      query_texts = [query],
      n_results = 3
  )
  chunks = results['documents'][0]
  return "\n\n".join(chunks)

#Json Schema

rag_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": "Search the company internal documents (HR Policy, Product, Security, onboarding, engineering) to find the relevant information. Use this when the user asks about company policies, products, features or support.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query you want to find relevant document"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

available_tools = {
    "search_docs": search_docs
}
print("Rag tool defined")

Rag tool defined


In [29]:
def rag_agent(question, verbose= True):
    messages = [
        {
            "role":"system",
            "content": (
                "You are a helpful company assistance and you have access to internal documents "
                "including HR polices and Porduct, engineering , onboarding, security"
                "Use the search_docs tool to find answers from the company documents"
                "If the documents dont contian the answer, say so clearly."
                "Always base your answers on the rerieved documets if you decided to use the tool."
                "If tool use is not needed, specifcy that you are not using tool and asnwer the quesiton based on your knowledge."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    if verbose:
          print(f"\n{'═' * 60}")
          print(f"🧑 Question: {question}")
          print(f"{'─' * 60}")


    max_steps = 3
    for step in range(max_steps):
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages,
            tools=rag_tools,
        )

        choice = response.choices[0]

        # LLM is done
        if choice.finish_reason == "stop":
            if verbose:
                print(f"🤖 Answer: {choice.message.content}")
                print(f"{'═' * 60}")
            return choice.message.content

        # LLM wants to search
        if choice.message.tool_calls:
            messages.append(choice.message)

            for tool_call in choice.message.tool_calls:
                func_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"  🔍 Searching: \"{args['query']}\"")

                result = available_tools[func_name](**args)

                if verbose:
                    print(f"  📄 Found {len(result)} chars of context")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })

    return "Could not answer within step limit."

print("✅ RAG Agent ready!")

✅ RAG Agent ready!


In [30]:
 # Test the RAG agent
rag_agent("How many sick leave days do I get per year?")


════════════════════════════════════════════════════════════
🧑 Question: How many sick leave days do I get per year?
────────────────────────────────────────────────────────────
  🔍 Searching: "sick leave days per year policy"
  📄 Found 961 chars of context
🤖 Answer: According to the company’s HR policy, employees are entitled to **12 sick leave days per year**. Sick leave beyond 3 consecutive days requires a medical certificate, and the leave cannot be carried forward or encashed.
════════════════════════════════════════════════════════════


'According to the company’s HR policy, employees are entitled to **12 sick leave days per year**. Sick leave beyond 3 consecutive days requires a medical certificate, and the leave cannot be carried forward or encashed.'

In [31]:
rag_agent("What is the capital of france?")


════════════════════════════════════════════════════════════
🧑 Question: What is the capital of france?
────────────────────────────────────────────────────────────
🤖 Answer: I’m not using the internal search tool for this question.  
The capital of France is **Paris**.
════════════════════════════════════════════════════════════


'I’m not using the internal search tool for this question.  \nThe capital of France is **Paris**.'